**Activity 1: Load and Review the Dataset**

In [58]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

df = pd.read_csv(Path(r"C:\Users\bhava\Downloads\ted_talks.csv\2020-05-01\ted_talks_es.csv"))

print("Dataset loaded successfully!")
print("Original dataset shape:", df.shape)

display(df.head())
print("\nColumn names:")
print(df.columns.tolist())

Dataset loaded successfully!
Original dataset shape: (3921, 19)


,talk_id,title,speaker_1,all_speakers,occupations,about_speakers,views,recorded_date,published_date,event,native_lang,available_lang,comments,duration,topics,related_talks,url,description,transcript
0,1,Al Gore sobre cómo evitar la crisis climática,Al Gore,{0: 'Al Gore'},{0: ['climate advocate']},{0: 'Nobel Laureate Al Gore focused the world’...,3523396,2006-02-25,2006-06-27,TED2006,en,"['ar', 'bg', 'cs', 'de', 'el', 'en', 'es', 'fa...",272.0,977,"['alternative energy', 'cars', 'climate change...","{243: 'New thinking on the climate crisis', 54...",https://www.ted.com/talks/al_gore_averting_the...,Con el mismo humor y humanidad que irradió en ...,Muchas gracias Chris. Y es en verdad un gran h...
1,7,"David Pogue dice ""La Simplicidad Vende""",David Pogue,{0: 'David Pogue'},{0: ['technology columnist']},{0: 'David Pogue is the personal technology co...,1920803,2006-02-24,2006-06-27,TED2006,en,"['ar', 'bg', 'de', 'el', 'en', 'es', 'fa', 'fr...",124.0,1286,"['computers', 'entertainment', 'interface desi...","{1725: '10 top time-saving tech tips', 2274: '...",https://www.ted.com/talks/david_pogue_simplici...,"El columnista del New York Times, David Pogue,...","Hola contestadora automática, mi vieja amiga. ..."
2,53,Un recorrido por la renovación urbana de la ma...,Majora Carter,{0: 'Majora Carter'},{0: ['activist for environmental justice']},{0: 'Majora Carter redefined the field of envi...,2664029,2006-02-26,2006-06-27,TED2006,en,"['ar', 'bg', 'bn', 'ca', 'cs', 'de', 'en', 'es...",219.0,1116,"['MacArthur grant', 'activism', 'business', 'c...",{1041: '3 stories of local eco-entrepreneurshi...,https://www.ted.com/talks/majora_carter_greeni...,"En una charla altamente emotiva, la activista ...","Si están presentes aquí hoy, y estoy muy conte..."
3,66,Ken Robinson dice que las escuelas matan la cr...,Sir Ken Robinson,{0: 'Sir Ken Robinson'},"{0: ['author', 'educator']}","{0: ""Creativity expert Sir Ken Robinson challe...",65052534,2006-02-25,2006-06-27,TED2006,en,"['af', 'ar', 'az', 'be', 'bg', 'bn', 'ca', 'cs...",4931.0,1164,"['children', 'creativity', 'culture', 'dance',...","{865: 'Bring on the learning revolution!', 173...",https://www.ted.com/talks/sir_ken_robinson_do_...,Sir Ken Robinson plantea de manera entretenida...,"Buenos días. ¿Cómo están? Ha sido increíble, ¿..."
4,92,Hans Rosling nos muestra las mejores estadísti...,Hans Rosling,{0: 'Hans Rosling'},{0: ['global health expert; data visionary']},"{0: 'In Hans Rosling’s hands, data sings. Glob...",14501766,2006-02-22,2006-06-27,TED2006,en,"['ar', 'az', 'bg', 'bn', 'bs', 'cs', 'da', 'de...",628.0,1190,"['Africa', 'Asia', 'Google', 'demo', 'economic...","{2056: ""Own your body's data"", 2296: 'A visual...",https://www.ted.com/talks/hans_rosling_the_bes...,Una manera única de presentar datos. Con la en...,"Hace unos 10 años, emprendí la tarea de enseña..."



Column names:
['talk_id', 'title', 'speaker_1', 'all_speakers', 'occupations', 'about_speakers', 'views', 'recorded_date', 'published_date', 'event', 'native_lang', 'available_lang', 'comments', 'duration', 'topics', 'related_talks', 'url', 'description', 'transcript']


In [60]:
print("Dataset information:")
df.info()

print("\nMissing values in each column:")
print(df.isna().sum())

print("\nExact duplicate rows:", df.duplicated().sum())

print("\nEmpty strings in text columns:")
for col in df.select_dtypes(include="object").columns:
    print(col, df[col].astype("string").str.strip().eq("").sum())

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3921 entries, 0 to 3920
Data columns (total 19 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   talk_id         3921 non-null   int64  
 1   title           3921 non-null   object 
 2   speaker_1       3921 non-null   object 
 3   all_speakers    3918 non-null   object 
 4   occupations     3409 non-null   object 
 5   about_speakers  3423 non-null   object 
 6   views           3921 non-null   int64  
 7   recorded_date   3920 non-null   object 
 8   published_date  3921 non-null   object 
 9   event           3921 non-null   object 
 10  native_lang     3921 non-null   object 
 11  available_lang  3921 non-null   object 
 12  comments        3306 non-null   float64
 13  duration        3921 non-null   int64  
 14  topics          3921 non-null   object 
 15  related_talks   3921 non-null   object 
 16  url             3921 non-null   object 
 17  description 

**Activity 2: Handle Missing Values**

In [61]:
missing_before = df.isna().sum()

print("Missing values before cleaning:")
display(missing_before[missing_before > 0].to_frame("missing_count"))

Missing values before cleaning:


,missing_count
all_speakers,3
occupations,512
about_speakers,498
recorded_date,1
comments,615


In [62]:
# Identify available text columns
possible_text_columns = [
    "title", "tags", "description", "transcript"
]

text_columns = [
    col for col in possible_text_columns
    if col in df.columns
]

print("Text columns found:", text_columns)

# Preserve the original dataset in df
# Create separate cleaned versions of available text fields
def clean_text(value):
    if pd.isna(value):
        return ""

    text = str(value)
    text = text.replace("\u00a0", " ")
    text = re.sub(r"\s+", " ", text)
    text = text.strip()
    text = text.lower()

    return text

for col in text_columns:
    df[col + "_clean"] = df[col].apply(clean_text)

print("Text preprocessing completed.")
display(df.head())


Text columns found: ['title', 'description', 'transcript']
Text preprocessing completed.


,talk_id,title,speaker_1,all_speakers,occupations,about_speakers,views,recorded_date,published_date,event,...,comments,duration,topics,related_talks,url,description,transcript,title_clean,description_clean,transcript_clean
0,1,Al Gore sobre cómo evitar la crisis climática,Al Gore,{0: 'Al Gore'},{0: ['climate advocate']},{0: 'Nobel Laureate Al Gore focused the world’...,3523396,2006-02-25,2006-06-27,TED2006,...,272.0,977,"['alternative energy', 'cars', 'climate change...","{243: 'New thinking on the climate crisis', 54...",https://www.ted.com/talks/al_gore_averting_the...,Con el mismo humor y humanidad que irradió en ...,Muchas gracias Chris. Y es en verdad un gran h...,al gore sobre cómo evitar la crisis climática,con el mismo humor y humanidad que irradió en ...,muchas gracias chris. y es en verdad un gran h...
1,7,"David Pogue dice ""La Simplicidad Vende""",David Pogue,{0: 'David Pogue'},{0: ['technology columnist']},{0: 'David Pogue is the personal technology co...,1920803,2006-02-24,2006-06-27,TED2006,...,124.0,1286,"['computers', 'entertainment', 'interface desi...","{1725: '10 top time-saving tech tips', 2274: '...",https://www.ted.com/talks/david_pogue_simplici...,"El columnista del New York Times, David Pogue,...","Hola contestadora automática, mi vieja amiga. ...","david pogue dice ""la simplicidad vende""","el columnista del new york times, david pogue,...","hola contestadora automática, mi vieja amiga. ..."
2,53,Un recorrido por la renovación urbana de la ma...,Majora Carter,{0: 'Majora Carter'},{0: ['activist for environmental justice']},{0: 'Majora Carter redefined the field of envi...,2664029,2006-02-26,2006-06-27,TED2006,...,219.0,1116,"['MacArthur grant', 'activism', 'business', 'c...",{1041: '3 stories of local eco-entrepreneurshi...,https://www.ted.com/talks/majora_carter_greeni...,"En una charla altamente emotiva, la activista ...","Si están presentes aquí hoy, y estoy muy conte...",un recorrido por la renovación urbana de la ma...,"en una charla altamente emotiva, la activista ...","si están presentes aquí hoy, y estoy muy conte..."
3,66,Ken Robinson dice que las escuelas matan la cr...,Sir Ken Robinson,{0: 'Sir Ken Robinson'},"{0: ['author', 'educator']}","{0: ""Creativity expert Sir Ken Robinson challe...",65052534,2006-02-25,2006-06-27,TED2006,...,4931.0,1164,"['children', 'creativity', 'culture', 'dance',...","{865: 'Bring on the learning revolution!', 173...",https://www.ted.com/talks/sir_ken_robinson_do_...,Sir Ken Robinson plantea de manera entretenida...,"Buenos días. ¿Cómo están? Ha sido increíble, ¿...",ken robinson dice que las escuelas matan la cr...,sir ken robinson plantea de manera entretenida...,"buenos días. ¿cómo están? ha sido increíble, ¿..."
4,92,Hans Rosling nos muestra las mejores estadísti...,Hans Rosling,{0: 'Hans Rosling'},{0: ['global health expert; data visionary']},"{0: 'In Hans Rosling’s hands, data sings. Glob...",14501766,2006-02-22,2006-06-27,TED2006,...,628.0,1190,"['Africa', 'Asia', 'Google', 'demo', 'economic...","{2056: ""Own your body's data"", 2296: 'A visual...",https://www.ted.com/talks/hans_rosling_the_bes...,Una manera única de presentar datos. Con la en...,"Hace unos 10 años, emprendí la tarea de enseña...",hans rosling nos muestra las mejores estadísti...,una manera única de presentar datos. con la en...,"hace unos 10 años, emprendí la tarea de enseña..."


**Activity 3: Remove Duplicate Records**

In [63]:
original_row_count = len(df)

# Count exact duplicate rows before removal
exact_duplicates_before = int(df.duplicated().sum())

# Remove exact duplicate rows
df = df.drop_duplicates().copy()

rows_after_exact_deduplication = len(df)
exact_duplicates_removed = original_row_count - rows_after_exact_deduplication

print("Original row count:", original_row_count)
print("Exact duplicate rows found:", exact_duplicates_before)
print("Exact duplicate rows removed:", exact_duplicates_removed)
print("Rows after duplicate removal:", rows_after_exact_deduplication)

Original row count: 3921
Exact duplicate rows found: 0
Exact duplicate rows removed: 0
Rows after duplicate removal: 3921


In [64]:
# Find possible identifiers available in the dataset
identifier_candidates = [
    "talk_id", "id", "url", "talk_url"
]

identifier_columns = [
    col for col in identifier_candidates if col in df.columns
]

for col in identifier_columns:
    values = df[col].astype("string").str.strip()
    valid_values = values[values.notna() & values.ne("")]

    print(f"\nRepeated values in {col}:")
    print(valid_values[valid_values.duplicated(keep=False)].value_counts())


Repeated values in talk_id:
Series([], Name: count, dtype: Int64)

Repeated values in url:
Series([], Name: count, dtype: Int64)


In [65]:
# Run only after verifying that repeated URLs
# refer to the same talk.
if "url" in df.columns:
    valid_url = df["url"].notna() & df["url"].astype(str).str.strip().ne("")
    verified = df.loc[valid_url].drop_duplicates(
        subset=["url"], keep="first"
    )

    df = pd.concat(
        [verified, df.loc[~valid_url]],
        ignore_index=True
    )

print("Rows after verified URL deduplication:", len(df))

Rows after verified URL deduplication: 3921


In [66]:
# Use only the cleaned fields that exist
clean_columns = [
    col + "_clean" for col in text_columns
]

def combine_text(row):
    parts = []

    for col in clean_columns:
        value = row[col]
        if isinstance(value, str) and value.strip():
            parts.append(value.strip())

    return " ".join(parts)

df["combined_text"] = df.apply(combine_text, axis=1)

print("Combined text column created successfully!")
display(df[["combined_text"]].head())

Combined text column created successfully!


,combined_text
0,al gore sobre cómo evitar la crisis climática ...
1,"david pogue dice ""la simplicidad vende"" el col..."
2,un recorrido por la renovación urbana de la ma...
3,ken robinson dice que las escuelas matan la cr...
4,hans rosling nos muestra las mejores estadísti...


In [67]:
print("BEFORE CLEANING")
print("Rows:", original_row_count)
print("Exact duplicate rows:", exact_duplicates_before)

print("\nAFTER CLEANING")
print("Rows:", len(df))
print("Exact duplicate rows:", int(df.duplicated().sum()))

print("\nMissing values after cleaning:")
display(df.isna().sum().to_frame("missing_count"))

print("\nCombined text examples:")
display(df[["combined_text"]].head())

BEFORE CLEANING
Rows: 3921
Exact duplicate rows: 0

AFTER CLEANING
Rows: 3921
Exact duplicate rows: 0

Missing values after cleaning:


,missing_count
talk_id,0
title,0
speaker_1,0
all_speakers,3
occupations,512
about_speakers,498
views,0
recorded_date,1
published_date,0
event,0



Combined text examples:


,combined_text
0,al gore sobre cómo evitar la crisis climática ...
1,"david pogue dice ""la simplicidad vende"" el col..."
2,un recorrido por la renovación urbana de la ma...
3,ken robinson dice que las escuelas matan la cr...
4,hans rosling nos muestra las mejores estadísti...
